# Dunnhumby Retail Analytics

## 1. Data Quality Analysis

Before performing exploratory analysis or building machine learning models,
we need to understand the quality of the raw data.

Poor data quality can lead to incorrect business conclusions and unreliable
machine learning models.

In this notebook, we will examine:

- Missing values
- Duplicate records
- Data types
- Invalid values
- Outliers
- Identifier consistency
- Relationships between datasets

The goal is to identify potential data quality issues and decide how each
issue should be handled.

## 2. Load the Raw Datasets

We will load the raw CSV files again so that this notebook can be executed
independently from the previous notebook.

No data will be modified at this stage.

In [ ]:
from pathlib import Path
import pandas as pd

# Define the raw data directory
RAW_DIR = Path("../01_data/01_raw")

# Find all CSV files
csv_files = sorted(RAW_DIR.glob("*.csv"))

# Load all datasets
datasets = {}

for file in csv_files:
    datasets[file.stem] = pd.read_csv(file)

print(f"Loaded {len(datasets)} datasets.")

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded 8 datasets.
campaign_desc: 30 rows × 4 columns
campaign_table: 7,208 rows × 3 columns
causal_data: 36,786,524 rows × 5 columns
coupon: 124,548 rows × 3 columns
coupon_redempt: 2,318 rows × 4 columns
hh_demographic: 801 rows × 8 columns
product: 92,353 rows × 7 columns
transaction_data: 2,595,732 rows × 12 columns


## 3. Missing Values

Missing values can affect both business analysis and machine learning.

However, missing data does not automatically mean that a dataset is
incorrect.

For example, demographic information may only be available for a subset
of households.

Therefore, we will first identify missing values and then determine
whether they represent a real data quality issue or an expected property
of the dataset.

In [11]:
missing_summary = []

for name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()

        if missing_count > 0:
            missing_summary.append({
                "dataset": name,
                "column": column,
                "missing_values": int(missing_count),
                "missing_percentage": round(
                    missing_count / len(df) * 100,
                    2
                )
            })

missing_summary = pd.DataFrame(missing_summary)

if missing_summary.empty:
    print("No missing values were found.")
else:
    display(
        missing_summary.sort_values(
            "missing_percentage",
            ascending=False
        )
    )

No missing values were found.


## 4. Duplicate Records

Duplicate rows can distort aggregations such as revenue, transaction count,
and customer frequency.

We will check for exact duplicate rows in every dataset.

A duplicate does not necessarily mean that the record is incorrect,
but it should be investigated before removing anything.

In [12]:
duplicate_summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "duplicate_rows": int(df.duplicated().sum()),
        "duplicate_percentage": round(
            df.duplicated().mean() * 100,
            2
        )
    }
    for name, df in datasets.items()
])

display(
    duplicate_summary.sort_values(
        "duplicate_rows",
        ascending=False
    )
)

,dataset,rows,duplicate_rows,duplicate_percentage
3,coupon,124548,5164,4.15
0,campaign_desc,30,0,0.00
1,campaign_table,7208,0,0.00
2,causal_data,36786524,0,0.00
4,coupon_redempt,2318,0,0.00
5,hh_demographic,801,0,0.00
6,product,92353,0,0.00
7,transaction_data,2595732,0,0.00


## 5. Numeric Data Validation

Numeric variables are particularly important for the transaction dataset.

We will inspect the minimum, maximum, mean, median, and standard deviation
of the main numeric variables.

This can help us identify unexpected values and potential outliers.

In [16]:
transactions = datasets["transaction_data"]

numeric_columns = [
    "QUANTITY",
    "SALES_VALUE",
    "RETAIL_DISC",
    "COUPON_DISC",
    "COUPON_MATCH_DISC"
]

numeric_summary = transactions[numeric_columns].describe().T

numeric_summary["median"] = transactions[numeric_columns].median()

display(numeric_summary)

,count,mean,std,min,25%,50%,75%,max,median
QUANTITY,2595732.0,100.428558,1153.436211,0.00,1.00,1.00,1.00,89638.00,1.00
SALES_VALUE,2595732.0,3.104120,4.182274,0.00,1.29,2.00,3.49,840.00,2.00
RETAIL_DISC,2595732.0,-0.538705,1.249191,-180.00,-0.69,-0.01,0.00,3.99,-0.01
COUPON_DISC,2595732.0,-0.016416,0.216841,-55.93,0.00,0.00,0.00,0.00,0.00
COUPON_MATCH_DISC,2595732.0,-0.002919,0.039690,-7.70,0.00,0.00,0.00,0.00,0.00


## 6. Negative Sales Values

Sales values should normally represent the amount paid by the customer.

However, negative values may occur because of returns, refunds, corrections,
or other transaction adjustments.

Therefore, negative sales values should be investigated rather than removed
automatically.

In [8]:
negative_sales = transactions[
    transactions["SALES_VALUE"] < 0
]

print(f"Negative sales records: {len(negative_sales):,}")
print(
    f"Negative sales value: "
    f"{negative_sales['SALES_VALUE'].sum():,.2f}"
)

display(
    negative_sales[
        [
            "household_key",
            "BASKET_ID",
            "DAY",
            "PRODUCT_ID",
            "QUANTITY",
            "SALES_VALUE"
        ]
    ].head(20)
)

Negative sales records: 0
Negative sales value: 0.00


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE


## 7. Quantity Validation

The `QUANTITY` variable requires special attention.

Large quantity values do not necessarily represent data errors.
Some products may be sold in bulk or by weight.

Instead of removing extreme values, we will first understand their
distribution and investigate the highest values.

In [9]:
quantity_summary = transactions["QUANTITY"].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]
)

display(quantity_summary.to_frame("QUANTITY"))

,QUANTITY
count,2.595732e+06
mean,1.004286e+02
std,1.153436e+03
min,0.000000e+00
50%,1.000000e+00
90%,2.000000e+00
95%,3.000000e+00
99%,1.000000e+01
99.9%,1.694354e+04
max,8.963800e+04


In [10]:
top_quantities = (
    transactions[
        [
            "household_key",
            "BASKET_ID",
            "PRODUCT_ID",
            "QUANTITY",
            "SALES_VALUE"
        ]
    ]
    .sort_values("QUANTITY", ascending=False)
    .head(20)
)

display(top_quantities)

,household_key,BASKET_ID,PRODUCT_ID,QUANTITY,SALES_VALUE
1750942,630,34749153595,6534178,89638,250.00
468356,2407,29392047893,6544236,85055,210.00
481876,630,29484790880,6534178,61335,150.21
166536,149,28210551971,6534178,51912,110.00
1340882,193,32956767959,6534178,48073,121.10
2472791,2133,41904760458,6534178,45475,100.00
1560340,149,33768630428,6534178,41833,115.00
376686,149,29035716247,6534178,41686,100.00
156049,1406,28167562655,6544236,41485,85.00
87625,107,27865225627,6534178,39365,72.00
